In [7]:
import pandas as pd

patients = pd.read_csv("patients.csv")
services = pd.read_csv("services_weekly.csv")
staff = pd.read_csv("staff.csv")
schedule = pd.read_csv("staff_schedule.csv")

print(patients.shape)
print(services.shape)
print(staff.shape)
print(schedule.shape)



(1000, 7)
(208, 10)
(110, 4)
(6552, 6)


In [9]:
print(patients.head())
print(patients.dtypes)
print(patients.isnull().sum())

     patient_id               name  age arrival_date departure_date  \
0  PAT-09484753  Richard Rodriguez   24   16-03-2025     22-03-2025   
1  PAT-f0644084     Shannon Walker    6   13-12-2025     14-12-2025   
2  PAT-ac6162e4       Julia Torres   24   29-06-2025     05-07-2025   
3  PAT-3dda2bb5    Crystal Johnson   32   12-10-2025     23-10-2025   
4  PAT-08591375        Garrett Lin   25   18-02-2025     25-02-2025   

            service  satisfaction  
0           surgery            61  
1           surgery            83  
2  general_medicine            83  
3         emergency            81  
4               ICU            76  
patient_id          str
name                str
age               int64
arrival_date        str
departure_date      str
service             str
satisfaction      int64
dtype: object
patient_id        0
name              0
age               0
arrival_date      0
departure_date    0
service           0
satisfaction      0
dtype: int64


In [11]:
patients['arrival_date'] = pd.to_datetime(patients['arrival_date'], format='%d-%m-%Y')
patients['departure_date'] = pd.to_datetime(patients['departure_date'], format='%d-%m-%Y')

patients['length_of_stay'] = (patients['departure_date'] - patients['arrival_date']).dt.days

print(patients[['arrival_date','departure_date','length_of_stay']].head())
print(patients['length_of_stay'].describe())

  arrival_date departure_date  length_of_stay
0   2025-03-16     2025-03-22               6
1   2025-12-13     2025-12-14               1
2   2025-06-29     2025-07-05               6
3   2025-10-12     2025-10-23              11
4   2025-02-18     2025-02-25               7
count    1000.000000
mean        7.407000
std         3.953857
min         1.000000
25%         4.000000
50%         7.000000
75%        11.000000
max        14.000000
Name: length_of_stay, dtype: float64


In [15]:
print(patients.duplicated().sum())
print(patients['service'].unique())
print(patients['age'].describe())
print(patients['satisfaction'].describe())

0
<StringArray>
['surgery', 'general_medicine', 'emergency', 'ICU']
Length: 4, dtype: str
count    1000.000000
mean       45.337000
std        25.999912
min         0.000000
25%        23.000000
50%        46.000000
75%        68.000000
max        89.000000
Name: age, dtype: float64
count    1000.000000
mean       79.597000
std        11.550325
min        60.000000
25%        70.000000
50%        80.000000
75%        89.250000
max        99.000000
Name: satisfaction, dtype: float64


In [17]:
print(services.dtypes)
print(services.isnull().sum())
print(services['service'].unique())
print(services['event'].unique())

week                    int64
month                   int64
service                   str
available_beds          int64
patients_request        int64
patients_admitted       int64
patients_refused        int64
patient_satisfaction    int64
staff_morale            int64
event                     str
dtype: object
week                    0
month                   0
service                 0
available_beds          0
patients_request        0
patients_admitted       0
patients_refused        0
patient_satisfaction    0
staff_morale            0
event                   0
dtype: int64
<StringArray>
['emergency', 'surgery', 'general_medicine', 'ICU']
Length: 4, dtype: str
<StringArray>
['none', 'flu', 'donation', 'strike']
Length: 4, dtype: str


In [19]:
print(staff.dtypes)
print(staff.isnull().sum())
print(staff['role'].unique())
print(staff['service'].unique())

print(schedule.dtypes)
print(schedule.isnull().sum())
print(schedule['present'].unique())

staff_id      str
staff_name    str
role          str
service       str
dtype: object
staff_id      0
staff_name    0
role          0
service       0
dtype: int64
<StringArray>
['doctor', 'nurse', 'nursing_assistant']
Length: 3, dtype: str
<StringArray>
['emergency', 'surgery', 'general_medicine', 'ICU']
Length: 4, dtype: str
week          int64
staff_id        str
staff_name      str
role            str
service         str
present       int64
dtype: object
week          0
staff_id      0
staff_name    0
role          0
service       0
present       0
dtype: int64
[1 0]


In [21]:
service_stats = patients.groupby('service').agg(
    avg_length_of_stay=('length_of_stay', 'mean'),
    avg_satisfaction=('satisfaction', 'mean'),
    patient_count=('patient_id', 'count')
).sort_values('avg_length_of_stay', ascending=False)

print(service_stats)

                  avg_length_of_stay  avg_satisfaction  patient_count
service                                                              
surgery                     7.866142         80.314961            254
ICU                         7.605809         79.921162            241
emergency                   7.159696         79.547529            263
general_medicine            6.995868         78.574380            242


In [23]:
correlation = patients['length_of_stay'].corr(patients['satisfaction'])
print("Correlation:", correlation)

Correlation: 0.07349463713698824


In [25]:
age_corr_satisfaction = patients['age'].corr(patients['satisfaction'])
age_corr_stay = patients['age'].corr(patients['length_of_stay'])
print("Age vs Satisfaction:", age_corr_satisfaction)
print("Age vs Length of Stay:", age_corr_stay)

Age vs Satisfaction: -0.05622929563225173
Age vs Length of Stay: -0.050149005919341266


In [27]:
services['refusal_rate'] = services['patients_refused'] / services['patients_request']

refusal_by_service = services.groupby('service')['refusal_rate'].mean().sort_values(ascending=False)
print(refusal_by_service)

service
emergency           0.767689
general_medicine    0.346211
surgery             0.172578
ICU                 0.116615
Name: refusal_rate, dtype: float64


In [29]:
event_impact = services.groupby('event').agg(
    avg_requests=('patients_request', 'mean'),
    avg_admitted=('patients_admitted', 'mean'),
    avg_refused=('patients_refused', 'mean'),
    avg_satisfaction=('patient_satisfaction', 'mean')
)
print(event_impact)

          avg_requests  avg_admitted  avg_refused  avg_satisfaction
event                                                              
donation     54.714286     25.571429    29.142857         82.714286
flu         161.263158     35.105263   126.157895         78.736842
none         56.201220     27.823171    28.378049         79.725610
strike       40.545455     23.909091    16.636364         82.818182


In [31]:
morale_corr = services['staff_morale'].corr(services['patient_satisfaction'])
print("Staff morale vs Patient satisfaction:", morale_corr)

Staff morale vs Patient satisfaction: 0.008258651068417512


In [33]:
weekly_attendance = schedule.groupby(['week', 'service'])['present'].sum().reset_index()
weekly_attendance.rename(columns={'present': 'staff_present'}, inplace=True)

merged = services.merge(weekly_attendance, on=['week', 'service'])
merged['patients_per_staff'] = merged['patients_admitted'] / merged['staff_present']

staffing_ratio = merged.groupby('service')['patients_per_staff'].mean().sort_values(ascending=False)
print(staffing_ratio)

service
ICU                 inf
emergency           inf
general_medicine    inf
surgery             inf
Name: patients_per_staff, dtype: float64


In [39]:
print(weekly_attendance['staff_present'].describe())
print(weekly_attendance.head(10))

count    208.000000
mean      18.894231
std       13.892909
min        0.000000
25%        0.000000
50%       23.500000
75%       30.250000
max       38.000000
Name: staff_present, dtype: float64
   week           service  staff_present
0     1               ICU             31
1     1         emergency             35
2     1  general_medicine             27
3     1           surgery             23
4     2               ICU             30
5     2         emergency             37
6     2  general_medicine             25
7     2           surgery             23
8     3               ICU              0
9     3         emergency              0


In [41]:
merged_clean = merged[merged['staff_present'] > 0]

staffing_ratio = merged_clean.groupby('service')['patients_per_staff'].mean().sort_values(ascending=False)
print(staffing_ratio)

service
general_medicine    1.792677
surgery             1.490692
emergency           0.635838
ICU                 0.425094
Name: patients_per_staff, dtype: float64


In [43]:
weekly_attendance_with_event = services[['week','event']].drop_duplicates().merge(weekly_attendance, on='week')

attendance_by_event = weekly_attendance_with_event.groupby('event')['staff_present'].mean().sort_values(ascending=False)
print(attendance_by_event)

event
none        18.894231
donation    18.142857
strike      17.795455
flu         15.839286
Name: staff_present, dtype: float64


In [44]:
patients.to_csv('patients_clean.csv', index=False)
services.to_csv('services_weekly_clean.csv', index=False)
merged_clean.to_csv('staffing_merged_clean.csv', index=False)

print("Exported successfully")

Exported successfully
